# dmpbridge — Experiment Log

Records current model experiments, prompting strategies, and results.

**Dataset:** 10 manually labeled DMP documents, 741 blocks total  
**Evaluation:** Block-level label accuracy and per-label F1 (see per-model eval notebooks in `notebooks/`)  
**Labels:** `title`, `section.title`, `section.description`, `question.text`, `answer.text`  
**Output directory:** `data/output/labeled/`

## Experiment Registry

| ID | Model | Strategy | Output tag | Samples | Status |
|---|---|---|---|---|---|
| M03 | Llama 3.3 70B | batch | `llama3.3-70b_batch` | 10/10 | Done |
| M04 | Llama 3.3 70B | whole-doc | `llama3.3-70b_whole_doc` | 10/10 | Done |
| M05 | Llama 3.1 8B | batch | `llama3.1-8b_batch` | 10/10 | Done |
| M06 | Llama 3.1 8B | whole-doc | `llama3.1-8b_whole_doc` | 10/10 | Done |
| M07 | Llama 3.3 8B | batch | `llama3.3-8b_batch` | — | Pending |
| M08 | Llama 3.3 8B | whole-doc | `llama3.3-8b_whole_doc` | — | Pending |
| M09 | Gemma 4 4B | batch | `gemma4-4b_batch` | — | Pending |
| M10 | Gemma 4 4B | whole-doc | `gemma4-4b_whole_doc` | — | Pending |
| F01 | Llama 3.1 8B fine-tune | — | — | — | Planned |

Output files: `data/output/labeled/{tag}/sampleN.json`  
Detailed analysis: `notebooks/003_strategy_comparison.ipynb` and per-model eval notebooks.

In [ ]:
"""Live accuracy + per-label F1 summary for all registered experiments."""
from dmpbridge.evaluation.evaluate import load_method, compute_f1_rows, LABELS, SHORT

EXPERIMENTS = {
    "M03 Llama 3.3 70B batch":     "llama3.3-70b_batch",
    "M04 Llama 3.3 70B whole-doc": "llama3.3-70b_whole_doc",
    "M05 Llama 3.1 8B batch":      "llama3.1-8b_batch",
    "M06 Llama 3.1 8B whole-doc":  "llama3.1-8b_whole_doc",
    "M07 Llama 3.3 8B batch":      "llama3.3-8b_batch",
    "M08 Llama 3.3 8B whole-doc":  "llama3.3-8b_whole_doc",
    "M09 Gemma 4 4B batch":        "gemma4-4b_batch",
    "M10 Gemma 4 4B whole-doc":    "gemma4-4b_whole_doc",
}

hdr = f"{'Experiment':<30}  {'Accuracy':>9}" + "".join(f"  {s:>10}" for s in SHORT)
print(hdr)
print("-" * len(hdr))
for name, tag in EXPERIMENTS.items():
    df, conf, _ = load_method(tag)
    if df is None:
        print(f"  {name:<30}  {'N/A':>9}  (no output files — run the experiment first)")
        continue
    tc, tn = int(df["correct"].sum()), int(df["total"].sum())
    f1_by_label = compute_f1_rows(conf).set_index("label")["f1"]
    f1_str = "".join(f"  {f1_by_label.get(lbl, 0)*100:>9.1f}%" for lbl in LABELS)
    print(f"  {name:<30}  {tc/tn*100:>8.1f}%{f1_str}")

---
## Prompting Strategies

### Batch

The document is classified in overlapping windows of 10 blocks. Each window carries 3 blocks of context from the previous window to preserve label continuity across boundaries. The model makes one API call per window.

- Context window: 10 blocks, 3-block overlap
- API calls per document: approximately total blocks / 7
- Provider: Ollama (local) or Anthropic API
- Inference: `dmpbridge-experiment experiments/{name}-batch.yaml`

### Whole-document

All extracted blocks from the document are passed to the model in a single API call. The model classifies the entire document at once, with full visibility into structure and context.

- Context window: full document (typically 60–100 blocks)
- API calls per document: 1
- Provider: Ollama (local) or Anthropic API
- Inference: `dmpbridge-wholedoc` CLI

### PDF-direct

The raw PDF is sent directly to a vision-capable model. No pdfplumber extraction — the model reads the PDF and classifies paragraph-level blocks in a single call.

- Context window: full PDF (raw bytes)
- API calls per document: 1
- Provider: Ollama (local) or any vision-capable model
- Block granularity: paragraph-level (~20 blocks per document vs ~74 for pdfplumber)
- Inference: `dmpbridge-pdf --model <vision-model>`



---
## Llama 3.3 70B

**Provider:** Ollama (local)  
**Model ID:** `llama3.3:70b` (Q4_K_M, ~42 GB)  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/005_llama3.3-70b.ipynb`

### M03 — Batch

**Output files:** `data/output/labeled/llama3.3-70b_batch/sampleN.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 94.1% (697/741) |
| Samples complete | 10/10 |

Best performing free local model. Runs fully offline. Detailed per-label F1 in eval notebook.

### M04 — Whole-document

**Output files:** `data/output/labeled/llama3.3-70b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 91.8% (680/741) |
| Samples complete | 10/10 |
| Delta vs batch | −2.3pp |

Whole-document prompting hurts for this model. With full document context, the model over-classifies `question.text` blocks as `section.description`, producing a large drop in `question.text` F1 (−36pp). The batch strategy is the recommended approach for Llama 3.3 70B.

---
## Llama 3.1 8B

**Provider:** Ollama (local)  
**Model ID:** `llama3.1:8b` (Q4_K_M, ~5 GB)  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/004_llama3.1-8b.ipynb`

### M05 — Batch

**Output files:** `data/output/labeled/llama3.1-8b_batch/sampleN.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 67.3% (499/741) |
| Samples complete | 10/10 |

Substantially below the 70B model. The 8B model cannot reliably distinguish `section.description` (funder-written) from `question.text` (researcher-written) in batch mode — few-shot examples are not sufficient to bridge this gap at this model size.

### M06 — Whole-document

**Output files:** `data/output/labeled/llama3.1-8b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 84.2% (624/741) |
| Samples complete | 10/10 |
| Delta vs batch | +16.9pp |

Whole-document context produces a large accuracy gain for the 8B model (+16.9pp overall, +22pp `question.text` F1). Full document visibility compensates for the model's weaker semantic discrimination in batch mode. Even so, 84.2% is still 10pp below Llama 3.3 70B batch and is not suitable for production use without fine-tuning.

---
## Llama 3.3 8B

**Provider:** Ollama (local)  
**Model ID:** `llama3.3:8b`  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/006_llama3.3-8b.ipynb`

### M07 — Batch

**Output files:** `data/output/labeled/llama3.3-8b_batch/sampleN.json`  
**Inference:** `dmpbridge-experiment experiments/llama3.3-8b-batch.yaml`

| Metric | Value |
|---|---|
| Overall accuracy | Pending |
| Samples complete | 0/10 |

### M08 — Whole-document

**Output files:** `data/output/labeled/llama3.3-8b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model llama3.3:8b`

| Metric | Value |
|---|---|
| Overall accuracy | Pending |
| Samples complete | 0/10 |

---
## Gemma 4 4B

**Provider:** Ollama (local)  
**Model ID:** `gemma4:4b`  
**Input:** `data/input/pdfs/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/007_gemma4-4b.ipynb`

### M09 — Batch

**Output files:** `data/output/labeled/gemma4-4b_batch/sampleN.json`  
**Inference:** `dmpbridge-experiment experiments/gemma4-4b-batch.yaml`

| Metric | Value |
|---|---|
| Overall accuracy | Pending |
| Samples complete | 0/10 |

### M10 — Whole-document

**Output files:** `data/output/labeled/gemma4-4b_whole_doc/sampleN.json`  
**Inference:** `dmpbridge-wholedoc --provider ollama --model gemma4:4b`

| Metric | Value |
|---|---|
| Overall accuracy | Pending |
| Samples complete | 0/10 |

---
## Cross-model Summary

| Model | Batch | Whole-doc | Best | Recommendation |
|---|---|---|---|---|
| Llama 3.3 70B | **94.1%** | 91.8% | Batch | Use batch |
| Llama 3.1 8B | 67.3% | **84.2%** | Whole-doc | Neither — see F01 |
| Llama 3.3 8B | Pending | Pending | — | — |
| Gemma 4 4B | Pending | Pending | — | — |

**Key findings (existing models):**

- Whole-document context helps small models (+16pp for Llama 3.1 8B) but slightly hurts large models (−2pp for Llama 3.3 70B).
- Llama 3.3 70B batch (93.8%) is the best performing model — runs fully offline at no cost.

**Hypothesis for new models:**

- Llama 3.3 8B: expected to outperform Llama 3.1 8B (same size, newer architecture) — target >75% batch.
- Gemma 4 4B: smallest model in the suite; target >60% batch.

Detailed per-label F1 and per-sample breakdowns: `notebooks/003_strategy_comparison.ipynb`.

---
## Future Plans

### F01 — Fine-tune Llama 3.1 8B

**Status:** Planned  
**Depends on:** Expanding the labeled dataset to at least 20 samples (need held-out evaluation set)

**Hypothesis:** Fine-tuning on labeled DMP blocks will close most of the accuracy gap between Llama 3.1 8B and Llama 3.3 70B, making low-resource deployment viable.

**Planned approach:**
- Generate block-level training pairs from `data/input/ground_truth/`
- Fine-tune with LoRA or QLoRA
- Evaluate on a held-out set not used in training
- Target: `question.text` F1 from 33% (batch) to 70%+

**Blocker:** The current 10-sample dataset is too small to split into train/eval. Dataset expansion is a prerequisite.

---

*Other possible future work: sequence correction post-processing (state machine over label sequence); dataset expansion to non-NIH/NSF funders.*